In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


YELLOW   = '#FAEDCB'
MINT     = '#C9E4DE'
BLUE     = '#C6DEF1'
LAVENDER = '#DBCDF0'
PINK     = '#F2C6DE'
PEACH    = '#F7D9C4'
PASTEL   = [YELLOW, MINT, BLUE, LAVENDER, PINK, PEACH]
BG       = '#F7F4F0'
TEXT     = '#4A4A4A'
DARK_ACCENT = '#9B8EA8'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG,
    'font.family': 'DejaVu Sans', 'text.color': TEXT,
    'axes.labelcolor': TEXT, 'xtick.color': TEXT, 'ytick.color': TEXT,
    'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.titlecolor': TEXT,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.spines.left': False, 'axes.spines.bottom': False,
    'axes.grid': True, 'grid.color': '#E8E4E0', 'grid.linewidth': 0.8,
})

def pastel_bar(ax, x, y, colors=None, horizontal=False):
    if colors is None:
        colors = (PASTEL * 10)[:len(x)]
    if horizontal:
        bars = ax.barh(x, y, color=colors, height=0.6, edgecolor='white', linewidth=1.5)
    else:
        bars = ax.bar(x, y, color=colors, width=0.6, edgecolor='white', linewidth=1.5)
    return bars

def style_ax(ax, title):
    ax.set_title(title, pad=12, color=TEXT)
    ax.tick_params(labelsize=9)

print("✅ Libraries & palette loaded")

## 1. Load Data


In [ ]:
df = pd.read_csv('zomato_india.csv')
print("Shape:", df.shape)
df.head()

In [ ]:
print("Missing Values:")
print(df.isnull().sum())
print()
df[['rating','votes','cost_for_two']].describe()

## 2. Data Cleaning

In [ ]:
# Fill missing ratings with median
df['rating'].fillna(df['rating'].median(), inplace=True)
df['rating'] = df['rating'].round(1)

# Rating buckets
def rate_bucket(r):
    if r >= 4.5: return 'Excellent (4.5+)'
    elif r >= 4.0: return 'Good (4.0–4.4)'
    elif r >= 3.5: return 'Average (3.5–3.9)'
    else: return 'Below Avg (<3.5)'

df['rating_category'] = df['rating'].apply(rate_bucket)

# Price range buckets
df['price_range'] = pd.cut(df['cost_for_two'],
    bins=[0, 300, 700, 1500, 5000],
    labels=['Budget', 'Mid-Range', 'Premium', 'Luxury'])

print("✅ Cleaning done. Final shape:", df.shape)
df.head()

## 3. City-Level Insights

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('🍽️  Zomato India — City-Level Insights', fontsize=17, fontweight='bold', color=TEXT, y=1.01)
fig.patch.set_facecolor(BG)

city_counts = df['city'].value_counts()
pastel_bar(axes[0,0], city_counts.index, city_counts.values, colors=(PASTEL*2)[:len(city_counts)])
for i, v in enumerate(city_counts.values):
    axes[0,0].text(i, v+3, str(v), ha='center', fontsize=8.5, color=TEXT)
style_ax(axes[0,0], 'Restaurants by City')
axes[0,0].tick_params(axis='x', rotation=30)
axes[0,0].set_ylabel('Count', color=TEXT)

avg_rating = df.groupby('city')['rating'].mean().sort_values(ascending=True)
pastel_bar(axes[0,1], avg_rating.index, avg_rating.values, colors=(PASTEL*2)[:len(avg_rating)], horizontal=True)
for i, v in enumerate(avg_rating.values):
    axes[0,1].text(v+0.01, i, f'{v:.2f}', va='center', fontsize=8.5, color=TEXT)
style_ax(axes[0,1], 'Avg Rating by City')
axes[0,1].set_xlim(3.0, 4.3)

online_pct = df['online_order'].value_counts()
wedges, texts, autotexts = axes[1,0].pie(online_pct.values, labels=online_pct.index,
    autopct='%1.1f%%', colors=[PINK, MINT], startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=3), textprops={'fontsize': 10, 'color': TEXT})
for at in autotexts: at.set_color(TEXT)
style_ax(axes[1,0], 'Online Order Availability')

avg_cost = df.groupby('city')['cost_for_two'].mean().sort_values(ascending=False)
pastel_bar(axes[1,1], avg_cost.index, avg_cost.values, colors=(PASTEL*2)[:len(avg_cost)])
for i, v in enumerate(avg_cost.values):
    axes[1,1].text(i, v+8, f'₹{int(v)}', ha='center', fontsize=8, color=TEXT)
style_ax(axes[1,1], 'Avg Cost for Two by City (₹)')
axes[1,1].tick_params(axis='x', rotation=30)

plt.tight_layout(pad=2)
plt.show()

## 4. Cuisine & Restaurant Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('🍛  Cuisine & Restaurant Type', fontsize=17, fontweight='bold', color=TEXT)
fig.patch.set_facecolor(BG)

top_cuisines = df['cuisine'].value_counts().head(10)
pastel_bar(axes[0], top_cuisines.index[::-1], top_cuisines.values[::-1],
           colors=(PASTEL*2)[:10], horizontal=True)
for i, v in enumerate(top_cuisines.values[::-1]):
    axes[0].text(v+1, i, str(v), va='center', fontsize=9, color=TEXT)
style_ax(axes[0], 'Top 10 Cuisines')
axes[0].set_xlabel('Number of Restaurants', color=TEXT)

type_counts = df['restaurant_type'].value_counts()
wedges, texts, autotexts = axes[1].pie(type_counts.values, labels=type_counts.index,
    autopct='%1.0f%%', colors=(PASTEL*2)[:len(type_counts)], startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=2), textprops={'fontsize': 8, 'color': TEXT})
for at in autotexts: at.set_color(TEXT)
style_ax(axes[1], 'Restaurant Type Breakdown')

plt.tight_layout(pad=2)
plt.show()

## 5. Rating Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('⭐  Rating Distribution & Analysis', fontsize=17, fontweight='bold', color=TEXT)
fig.patch.set_facecolor(BG)

n, bins, patches = axes[0].hist(df['rating'], bins=20, edgecolor='white', linewidth=1)
for patch, color in zip(patches, (PASTEL*5)[:len(patches)]):
    patch.set_facecolor(color)
axes[0].axvline(df['rating'].mean(), color=DARK_ACCENT, linestyle='--', linewidth=2,
                label=f"Mean: {df['rating'].mean():.2f}")
axes[0].legend(fontsize=9)
style_ax(axes[0], 'Rating Distribution')
axes[0].set_xlabel('Rating', color=TEXT)
axes[0].set_ylabel('Frequency', color=TEXT)

cat_order = ['Excellent (4.5+)', 'Good (4.0–4.4)', 'Average (3.5–3.9)', 'Below Avg (<3.5)']
cat_counts = df['rating_category'].value_counts().reindex(cat_order, fill_value=0)
wedges, texts, autotexts = axes[1].pie(cat_counts.values, labels=cat_counts.index,
    autopct='%1.1f%%', colors=[MINT, BLUE, YELLOW, PINK], startangle=90,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=3),
    textprops={'fontsize': 8, 'color': TEXT})
for at in autotexts: at.set_color(TEXT)
style_ax(axes[1], 'Rating Categories')

top8 = df['cuisine'].value_counts().head(8).index
cuisine_rating = df[df['cuisine'].isin(top8)].groupby('cuisine')['rating'].mean().sort_values()
pastel_bar(axes[2], cuisine_rating.index, cuisine_rating.values,
           colors=(PASTEL*2)[:len(cuisine_rating)], horizontal=True)
for i, v in enumerate(cuisine_rating.values):
    axes[2].text(v+0.005, i, f'{v:.2f}', va='center', fontsize=9, color=TEXT)
style_ax(axes[2], 'Avg Rating by Cuisine (Top 8)')
axes[2].set_xlim(3.2, 4.2)

plt.tight_layout(pad=2)
plt.show()

## 6. Cost Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('💰  Cost & Price Range Analysis', fontsize=17, fontweight='bold', color=TEXT)
fig.patch.set_facecolor(BG)

n, bins, patches = axes[0].hist(df['cost_for_two'], bins=30, edgecolor='white', linewidth=1)
for patch, color in zip(patches, (PASTEL*5)[:len(patches)]):
    patch.set_facecolor(color)
axes[0].axvline(df['cost_for_two'].median(), color=DARK_ACCENT, linestyle='--', linewidth=2,
                label=f"Median: ₹{int(df['cost_for_two'].median())}")
axes[0].legend(fontsize=9)
style_ax(axes[0], 'Cost for Two Distribution')
axes[0].set_xlabel('Cost (₹)', color=TEXT)
axes[0].set_ylabel('Frequency', color=TEXT)

price_counts = df['price_range'].value_counts()
wedges, texts, autotexts = axes[1].pie(price_counts.values, labels=price_counts.index,
    autopct='%1.1f%%', colors=[YELLOW, PEACH, LAVENDER, PINK], startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=3), textprops={'fontsize': 9, 'color': TEXT})
for at in autotexts: at.set_color(TEXT)
style_ax(axes[1], 'Price Range Breakdown')

scatter = axes[2].scatter(df['cost_for_two'], df['rating'], alpha=0.5,
                          c=df['votes'], cmap='BuPu', s=35, edgecolors='white', linewidths=0.3)
cbar = plt.colorbar(scatter, ax=axes[2])
cbar.set_label('Votes', color=TEXT, fontsize=9)
style_ax(axes[2], 'Cost vs Rating (by Votes)')
axes[2].set_xlabel('Cost for Two (₹)', color=TEXT)
axes[2].set_ylabel('Rating', color=TEXT)

plt.tight_layout(pad=2)
plt.show()

## 7. City × Cuisine Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

pivot = df.pivot_table(values='rating', index='city', columns='cuisine', aggfunc='mean')
pivot = pivot[df['cuisine'].value_counts().head(8).index]

pastel_cmap = LinearSegmentedColormap.from_list('pastel_rg', [PINK, YELLOW, MINT], N=256)
sns.heatmap(pivot, annot=True, fmt='.2f', cmap=pastel_cmap,
            linewidths=2, linecolor=BG, vmin=3.2, vmax=4.2, ax=ax,
            annot_kws={'size': 9, 'color': TEXT},
            cbar_kws={'label': 'Avg Rating', 'shrink': 0.8})
ax.set_title('🗺️  Average Rating — City × Cuisine Heatmap', fontsize=14, fontweight='bold', color=TEXT, pad=15)
ax.set_xlabel('Cuisine', color=TEXT)
ax.set_ylabel('City', color=TEXT)

plt.tight_layout()
plt.show()

## 8. Business Insights

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('📊  Business Insights', fontsize=17, fontweight='bold', color=TEXT)
fig.patch.set_facecolor(BG)

online_rating = df.groupby('online_order')['rating'].mean()
pastel_bar(axes[0], online_rating.index, online_rating.values, colors=[MINT, PINK])
for i, v in enumerate(online_rating.values):
    axes[0].text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=10, fontweight='bold', color=TEXT)
style_ax(axes[0], 'Online Order Availability vs Avg Rating')
axes[0].set_ylim(3.4, 4.1)

table_rating = df.groupby('book_table')['rating'].mean()
pastel_bar(axes[1], table_rating.index, table_rating.values, colors=[LAVENDER, BLUE])
for i, v in enumerate(table_rating.values):
    axes[1].text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=10, fontweight='bold', color=TEXT)
style_ax(axes[1], 'Table Booking Option vs Avg Rating')
axes[1].set_ylim(3.4, 4.1)

plt.tight_layout(pad=2)
plt.show()

## 9. Key Business Insights

| # | Insight |
|---|---------|
| 1 | **Bangalore** has the most listed restaurants — most competitive market |
| 2 | **Mumbai** restaurants have the highest average rating |
| 3 | **66.6%** of restaurants offer online ordering — strong digital adoption |
| 4 | Restaurants with **table booking** tend to have higher ratings |
| 5 | **Median cost for two** is ₹353 — budget-to-mid-range market dominates |
| 6 | Cost and rating show weak positive correlation — price is not equal to quality |

## Conclusion

The Indian restaurant market is digitally evolving with online ordering becoming the norm. Bangalore leads in volume while Mumbai leads in quality perception. For Zomato, focusing on cities with lower average ratings could improve platform reputation and user satisfaction.
